# Sales Pipeline - EDA and Data Processing

This notebook contains the main sales pipeline for data exploration, transformation, and analysis.

This notebook builds a simple medallion-style pipeline (bronze → silver → gold) using Polars + DuckDB. In production this would correspond to PySpark tables in Databricks with Unity Catalog, orchestrated by Prefect.

In [183]:
# Dependency check
try:
    import polars as pl
    import duckdb
    from pathlib import Path

    print(f"✅ Polars version: {pl.__version__}")
    print(f"✅ DuckDB version: {duckdb.__version__}")
    print("\n🎉 All dependencies are ready!")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("\nPlease install packages with: python3 -m pip install polars duckdb")

✅ Polars version: 1.35.2
✅ DuckDB version: 1.4.2

🎉 All dependencies are ready!


In [184]:
from pathlib import Path
import polars as pl
import duckdb

# Find project root by navigating up from current directory until we find the 'data' folder
# This works regardless of where the notebook is run from
current = Path.cwd()
while current != current.parent:
    if (current / "data").exists() and (current / "data" / "countries.json").exists():
        ROOT_DIR = current
        break
    current = current.parent
else:
    # Fallback: assume we're in src/EDA and go up 2 levels
    ROOT_DIR = Path.cwd().parent.parent

DATA_DIR = ROOT_DIR / "data"
BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

# Make sure layer folders exist under the REAL data directory
for d in (BRONZE_DIR, SILVER_DIR, GOLD_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT_DIR}")
print(f"Data directory: {DATA_DIR}")
ROOT_DIR, DATA_DIR, BRONZE_DIR, SILVER_DIR, GOLD_DIR

Project root: /Users/dr.jenniferemberton/Documents/GitHub/sales_test
Data directory: /Users/dr.jenniferemberton/Documents/GitHub/sales_test/data


(PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test'),
 PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data'),
 PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/bronze'),
 PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/silver'),
 PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/gold'))

# Loader Helper

In [185]:
import json
from pathlib import Path
import polars as pl

def load_multi_json(path: Path) -> pl.DataFrame:
    """
    Load a file that contains many JSON objects concatenated together
    (e.g., {...}{...}{...}) into a Polars DataFrame.
    Works even if they are separated by commas/newlines/spaces.
    """
    text = path.read_text()
    records = []

    buf = []
    depth = 0
    in_obj = False

    for ch in text:
        if ch == "{":
            in_obj = True
            depth += 1
        if in_obj:
            buf.append(ch)
        if ch == "}":
            depth -= 1
            if depth == 0 and in_obj:
                obj_str = "".join(buf)
                records.append(json.loads(obj_str))
                buf = []
                in_obj = False

    return pl.DataFrame(records)


### Bronze Layer – Load Raw Data

In [186]:
# ---- Bronze layer: load raw JSON data ----

countries_raw = load_multi_json(DATA_DIR / "countries.json")
customers_raw = load_multi_json(DATA_DIR / "customers.json")
orders_raw    = load_multi_json(DATA_DIR / "orders.json")
products_raw  = load_multi_json(DATA_DIR / "products.json")
sales_raw     = load_multi_json(DATA_DIR / "sales.json")

# Peek at schemas / column names for later joins
for name, df in [
    ("countries", countries_raw),
    ("customers", customers_raw),
    ("orders", orders_raw),
    ("products", products_raw),
    ("sales", sales_raw),
]:
    print(f"\n{name} columns:", df.columns)
    display(df.head())


countries columns: ['Country', 'Currency', 'Name', 'Region', 'Population', 'Area (sq. mi.)', 'Pop. Density (per sq. mi.)', 'Coastline (coast per area ratio)', 'Net migration', 'Infant mortality (per 1000 births)', 'GDP ($ per capita)', 'Literacy (%)', 'Phones (per 1000)', 'Arable (%)', 'Crops (%)', 'Other (%)', 'Climate', 'Birthrate', 'Deathrate', 'Agriculture', 'Industry', 'Service']


Country,Currency,Name,Region,Population,Area (sq. mi.),Pop. Density (per sq. mi.),Coastline (coast per area ratio),Net migration,Infant mortality (per 1000 births),GDP ($ per capita),Literacy (%),Phones (per 1000),Arable (%),Crops (%),Other (%),Climate,Birthrate,Deathrate,Agriculture,Industry,Service
str,str,str,str,i64,i64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""AD""","""EUR""","""Andorra""","""WESTERN EUROPE""",71201,468,152.1,0.0,6.6,4.05,19000,100.0,497.2,2.22,0.0,97.78,3.0,8.71,6.25,null,null,null
"""AE""","""AED""","""United Arab Emirates""","""NEAR EAST""",2602713,82880,31.4,1.59,1.03,14.51,23200,77.9,475.3,0.6,2.25,97.15,1.0,18.96,4.4,0.04,0.585,0.375
"""AF""","""AFN""","""Afghanistan""","""ASIA (EX. NEAR EAST)""",31056997,647500,48.0,0.0,23.06,163.07,700,36.0,3.2,12.13,0.22,87.65,1.0,46.6,20.34,0.38,0.24,0.38
"""AG""","""XCD""","""Antigua & Barbuda""","""LATIN AMER. & CARIB""",69108,443,156.0,34.54,-6.15,19.46,11000,89.0,549.9,18.18,4.55,77.27,2.0,16.93,5.37,0.038,0.22,0.743
"""AI""","""XCD""","""Anguilla""","""LATIN AMER. & CARIB""",13477,102,132.1,59.8,10.76,21.03,8600,95.0,460.0,0.0,0.0,100.0,2.0,14.17,5.34,0.04,0.18,0.78



customers columns: ['CustomerId', 'Active', 'Name', 'Address', 'City', 'Country', 'Email']


CustomerId,Active,Name,Address,City,Country,Email
i64,bool,str,str,str,str,str
1,true,"""Jason Orr""","""Ap #387-8229 Nullam Road""","""Kilsyth""","""HN""","""lacus.Nulla@Classaptenttaciti.…"
2,true,"""Devin Herman""","""P.O. Box 905, 9608 Etiam St.""","""Portici""","""HR""","""Integer.vulputate.risus@est.co…"
3,true,"""Kennan Head""","""Ap #694-3226 Odio St.""","""Bothey""","""JO""","""ligula@porttitortellusnon.org"""
4,false,"""Peter Fry""","""Ap #378-2594 Arcu. Road""","""Oudegem""","""LR""","""eu@Phasellusvitaemauris.com"""
5,true,"""Guy Ball""","""418-8277 Sociis Ave""","""Charleville-Mézières""","""TG""","""vestibulum.Mauris.magna@ipsumd…"



orders columns: ['OrderId', 'CustomerId', 'Date']


OrderId,CustomerId,Date
i64,i64,str
1,181,"""2018-01-01"""
2,119,"""2018-01-01"""
3,69,"""2018-01-01"""
4,173,"""2018-01-01"""
5,237,"""2018-01-01"""



products columns: ['ProductId', 'Name', 'ManufacturedCountry', 'WeightGrams']


ProductId,Name,ManufacturedCountry,WeightGrams
i64,str,str,i64
1,"""Generac""","""GB""",55
2,"""Ambigue""","""US""",128
3,"""Vaguee""","""CA""",103
4,"""Dubioum""","""PR""",77
5,"""Fuzzi""","""SG""",83



sales columns: ['SaleId', 'OrderId', 'ProductId', 'Quantity']


SaleId,OrderId,ProductId,Quantity
i64,i64,i64,i64
1,1,3,1
2,1,7,9
3,2,5,12
4,2,10,3
5,2,4,3


#### Persist bronze layer to Parquet

In [187]:
# Persist raw data into bronze layer as Parquet

countries_raw.write_parquet(BRONZE_DIR / "countries.parquet")
customers_raw.write_parquet(BRONZE_DIR / "customers.parquet")
orders_raw.write_parquet(BRONZE_DIR / "orders.parquet")
products_raw.write_parquet(BRONZE_DIR / "products.parquet")
sales_raw.write_parquet(BRONZE_DIR / "sales.parquet")

BRONZE_DIR, list(BRONZE_DIR.glob("*.parquet"))


(PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/bronze'),
 [PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/bronze/products.parquet'),
  PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/bronze/orders.parquet'),
  PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/bronze/sales.parquet'),
  PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/bronze/customers.parquet'),
  PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/bronze/countries.parquet')])

## Silver Layer – Dimensional Model (Customers, Products, Countries, Fact Sales)

In [188]:
# Silver layer: read from bronze Parquet tables

countries_bronze = pl.read_parquet(BRONZE_DIR / "countries.parquet")
customers_bronze = pl.read_parquet(BRONZE_DIR / "customers.parquet")
orders_bronze    = pl.read_parquet(BRONZE_DIR / "orders.parquet")
products_bronze  = pl.read_parquet(BRONZE_DIR / "products.parquet")
sales_bronze     = pl.read_parquet(BRONZE_DIR / "sales.parquet")

countries_bronze.head(), customers_bronze.head(), orders_bronze.head()

(shape: (5, 22)
 ┌─────────┬──────────┬─────────────┬─────────────┬───┬───────────┬────────────┬──────────┬─────────┐
 │ Country ┆ Currency ┆ Name        ┆ Region      ┆ … ┆ Deathrate ┆ Agricultur ┆ Industry ┆ Service │
 │ ---     ┆ ---      ┆ ---         ┆ ---         ┆   ┆ ---       ┆ e          ┆ ---      ┆ ---     │
 │ str     ┆ str      ┆ str         ┆ str         ┆   ┆ f64       ┆ ---        ┆ f64      ┆ f64     │
 │         ┆          ┆             ┆             ┆   ┆           ┆ f64        ┆          ┆         │
 ╞═════════╪══════════╪═════════════╪═════════════╪═══╪═══════════╪════════════╪══════════╪═════════╡
 │ AD      ┆ EUR      ┆ Andorra     ┆ WESTERN     ┆ … ┆ 6.25      ┆ null       ┆ null     ┆ null    │
 │         ┆          ┆             ┆ EUROPE      ┆   ┆           ┆            ┆          ┆         │
 │ AE      ┆ AED      ┆ United Arab ┆ NEAR EAST   ┆ … ┆ 4.4       ┆ 0.04       ┆ 0.585    ┆ 0.375   │
 │         ┆          ┆ Emirates    ┆             ┆   ┆           

In [189]:
# -------------------------
# Silver dimensions
# -------------------------

# Country dimension: keep business-friendly geo info
dim_country = (
    countries_bronze
    .select(
        "Country",           # country code
        "Name",              # country name
        "Region",
        "Population",
        "GDP ($ per capita)",
    )
    .rename({
        "Country": "CountryCode",
        "Name": "CountryName",
        "GDP ($ per capita)": "GdpPerCapita",
    })
    .sort("CountryCode")
)

# Customer dimension: rename fields to be more descriptive
dim_customer = (
    customers_bronze
    .select(
        "CustomerId",
        "Active",
        "Name",
        "Address",
        "City",
        "Country",   
        "Email",
    )
    .rename({
        "Name": "CustomerName",
        "Country": "CountryCode",
    })
    .sort("CustomerId")
)

# Product dimension
dim_product = (
    products_bronze
    .select(
        "ProductId",
        "Name",
        "ManufacturedCountry",
        "WeightGrams",
    )
    .rename({
        "Name": "ProductName",
    })
    .sort("ProductId")
)

print("dim_country:", dim_country.shape)
print("dim_customer:", dim_customer.shape)
print("dim_product:", dim_product.shape)

dim_country.head(), dim_customer.head(), dim_product.head()

dim_country: (224, 5)
dim_customer: (400, 7)
dim_product: (12, 4)


(shape: (5, 5)
 ┌─────────────┬──────────────────────┬──────────────────────┬────────────┬──────────────┐
 │ CountryCode ┆ CountryName          ┆ Region               ┆ Population ┆ GdpPerCapita │
 │ ---         ┆ ---                  ┆ ---                  ┆ ---        ┆ ---          │
 │ str         ┆ str                  ┆ str                  ┆ i64        ┆ i64          │
 ╞═════════════╪══════════════════════╪══════════════════════╪════════════╪══════════════╡
 │ AD          ┆ Andorra              ┆ WESTERN EUROPE       ┆ 71201      ┆ 19000        │
 │ AE          ┆ United Arab Emirates ┆ NEAR EAST            ┆ 2602713    ┆ 23200        │
 │ AF          ┆ Afghanistan          ┆ ASIA (EX. NEAR EAST) ┆ 31056997   ┆ 700          │
 │ AG          ┆ Antigua & Barbuda    ┆ LATIN AMER. & CARIB  ┆ 69108      ┆ 11000        │
 │ AI          ┆ Anguilla             ┆ LATIN AMER. & CARIB  ┆ 13477      ┆ 8600         │
 └─────────────┴──────────────────────┴──────────────────────┴────────────┴

In [190]:
# -------------------------
# Clean orders: add OrderDate
# -------------------------
orders_clean = (
    orders_bronze
    .with_columns(
        pl.col("Date")
          .str.strptime(pl.Date, "%Y-%m-%d") 
          .alias("OrderDate")
    )
    .drop("Date")
)

# -------------------------
# Silver fact table: fact_sales_enriched
# -------------------------

fact_sales_enriched = (
    sales_bronze
    # Add order info (OrderDate, CustomerId)
    .join(orders_clean, on="OrderId", how="left")
    # Add customer info (uses CountryCode from dim_customer)
    .join(
        dim_customer.select(
            "CustomerId",
            "CustomerName",
            "City",
            "CountryCode",
            "Active",
        ),
        on="CustomerId",
        how="left",
    )
    # Add product info
    .join(
        dim_product.select(
            "ProductId",
            "ProductName",
            "ManufacturedCountry",
            "WeightGrams",
        ),
        on="ProductId",
        how="left",
    )
    # Add country / region info via CountryCode
    .join(
        dim_country.select(
            "CountryCode",
            "CountryName",
            "Region",
            "GdpPerCapita",
        ),
        on="CountryCode",
        how="left",
    )
    .select(
        "SaleId",
        "OrderId",
        "OrderDate",
        "CustomerId",
        "CustomerName",
        "City",
        "CountryCode",
        "CountryName",
        "Region",
        "GdpPerCapita",
        "ProductId",
        "ProductName",
        "ManufacturedCountry",
        "WeightGrams",
        "Quantity",
        "Active",          # is the customer active?
    )
    .sort(["OrderDate", "SaleId"])
)

print("fact_sales_enriched:", fact_sales_enriched.shape)
fact_sales_enriched.head()



fact_sales_enriched: (61883, 16)


SaleId,OrderId,OrderDate,CustomerId,CustomerName,City,CountryCode,CountryName,Region,GdpPerCapita,ProductId,ProductName,ManufacturedCountry,WeightGrams,Quantity,Active
i64,i64,date,i64,str,str,str,str,str,i64,i64,str,str,i64,i64,bool
1,1,2018-01-01,181,"""Peter Diaz""","""Camrose""","""MX""","""Mexico""","""LATIN AMER. & CARIB""",9000,3,"""Vaguee""","""CA""",103,1,true
2,1,2018-01-01,181,"""Peter Diaz""","""Camrose""","""MX""","""Mexico""","""LATIN AMER. & CARIB""",9000,7,"""Nebuloon""","""CH""",118,9,true
3,2,2018-01-01,119,"""Stewart Cain""","""Maria""","""AZ""","""Azerbaijan""","""C.W. OF IND. STATES""",3400,5,"""Fuzzi""","""SG""",83,12,true
4,2,2018-01-01,119,"""Stewart Cain""","""Maria""","""AZ""","""Azerbaijan""","""C.W. OF IND. STATES""",3400,10,"""Uncleary""","""PE""",93,3,true
5,2,2018-01-01,119,"""Stewart Cain""","""Maria""","""AZ""","""Azerbaijan""","""C.W. OF IND. STATES""",3400,4,"""Dubioum""","""PR""",77,3,true


In [191]:
# -------------------------
# Persist silver layer to Parquet
# -------------------------

dim_country.write_parquet(SILVER_DIR / "dim_country.parquet")
dim_customer.write_parquet(SILVER_DIR / "dim_customer.parquet")
dim_product.write_parquet(SILVER_DIR / "dim_product.parquet")
fact_sales_enriched.write_parquet(SILVER_DIR / "fact_sales_enriched.parquet")

SILVER_DIR, list(SILVER_DIR.glob("*.parquet"))

(PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/silver'),
 [PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/silver/fact_sales_enriched.parquet'),
  PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/silver/dim_country.parquet'),
  PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/silver/dim_product.parquet'),
  PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/silver/dim_customer.parquet')])

## Gold Layer – Aggregates for Power BI

In [192]:
dir(fact)

['__add__',
 '__annotations__',
 '__array__',
 '__arrow_c_stream__',
 '__bool__',
 '__class__',
 '__contains__',
 '__copy__',
 '__dataframe__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__floordiv__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__mod__',
 '__module__',
 '__mul__',
 '__ne__',
 '__new__',
 '__radd__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__reversed__',
 '__rmul__',
 '__setattr__',
 '__setitem__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__sub__',
 '__subclasshook__',
 '__truediv__',
 '__weakref__',
 '_accessors',
 '_cast_all_from_to',
 '_comp',
 '_compare_to_non_df',
 '_compare_to_other_df',
 '_df',
 '_div',
 '_from_arrow',
 '_from_pandas',
 '_from_pydf',
 '_import_columns',
 '_ipython_key_completions_',
 '_replace',
 '_repr_html_',
 '_row_encode',
 '_to_metadata',
 '_to_pa

In [193]:
# -------------------------
# Gold layer: aggregates for Power BI
# -------------------------

# Load silver fact
fact = pl.read_parquet(SILVER_DIR / "fact_sales_enriched.parquet")

print(fact.columns)   # optional sanity check

# 1) Sales by customer (units sold)
sales_by_customer = (
    fact
    .group_by(["CustomerId", "CustomerName", "City", "CountryName", "Region"])
    .agg([
        pl.col("Quantity").sum().alias("TotalQuantity"),
        pl.col("OrderId").n_unique().alias("DistinctOrders"),
    ])
    .sort("TotalQuantity", descending=True)
)

# 2) Sales by geography
sales_by_geo = (
    fact
    .group_by(["CountryName", "Region"])
    .agg([
        pl.col("Quantity").sum().alias("TotalQuantity"),
        pl.col("OrderId").n_unique().alias("DistinctOrders"),
    ])
    .sort("TotalQuantity", descending=True)
)

# 3) Sales over time
sales_over_time = (
    fact
    .group_by("OrderDate")
    .agg([
        pl.col("Quantity").sum().alias("TotalQuantity"),
        pl.col("OrderId").n_unique().alias("DistinctOrders"),
    ])
    .sort("OrderDate")
)

# 4) (Optional) Sales by product
sales_by_product = (
    fact
    .group_by(["ProductId", "ProductName", "ManufacturedCountry"])
    .agg([
        pl.col("Quantity").sum().alias("TotalQuantity"),
        pl.col("OrderId").n_unique().alias("DistinctOrders"),
    ])
    .sort("TotalQuantity", descending=True)
)

# -------------------------
# Persist gold to Parquet + CSV for Power BI
# -------------------------
sales_by_customer.write_parquet(GOLD_DIR / "sales_by_customer.parquet")
sales_by_geo.write_parquet(GOLD_DIR / "sales_by_geo.parquet")
sales_over_time.write_parquet(GOLD_DIR / "sales_over_time.parquet")
sales_by_product.write_parquet(GOLD_DIR / "sales_by_product.parquet")

sales_by_customer.write_csv(GOLD_DIR / "sales_by_customer.csv")
sales_by_geo.write_csv(GOLD_DIR / "sales_by_geo.csv")
sales_over_time.write_csv(GOLD_DIR / "sales_over_time.csv")
sales_by_product.write_csv(GOLD_DIR / "sales_by_product.csv")

sales_by_customer.head()

['SaleId', 'OrderId', 'OrderDate', 'CustomerId', 'CustomerName', 'City', 'CountryCode', 'CountryName', 'Region', 'GdpPerCapita', 'ProductId', 'ProductName', 'ManufacturedCountry', 'WeightGrams', 'Quantity', 'Active']


CustomerId,CustomerName,City,CountryName,Region,TotalQuantity,DistinctOrders
i64,str,str,str,str,i64,u32
71,"""Nicholas Craig""","""Zwettl-Niederösterreich""","""Papua New Guinea""","""OCEANIA""",1056,42
35,"""Jameson Glover""","""Belgaum""","""N. Mariana Islands""","""OCEANIA""",970,38
396,"""Gabriel Cabrera""","""Labico""","""Burundi""","""SUB-SAHARAN AFRICA""",930,42
236,"""Allistair Warner""","""Marcq-en-Baroeul""","""Kenya""","""SUB-SAHARAN AFRICA""",929,43
291,"""Demetrius Heath""","""Trochu""","""Italy""","""WESTERN EUROPE""",928,44


# My Tiny Warehouse 🏠

In [194]:
# -------------------------
# Register gold tables in DuckDB (tiny warehouse demo)
# -------------------------
db_path = DATA_DIR / "sales.duckdb"
print("DuckDB path:", db_path)

con = duckdb.connect(str(db_path))

con.execute(
    "CREATE OR REPLACE TABLE sales_by_customer AS "
    "SELECT * FROM read_parquet(?);",
    [str(GOLD_DIR / "sales_by_customer.parquet")]
)

con.execute(
    "CREATE OR REPLACE TABLE sales_by_geo AS "
    "SELECT * FROM read_parquet(?);",
    [str(GOLD_DIR / "sales_by_geo.parquet")]
)

con.execute(
    "CREATE OR REPLACE TABLE sales_over_time AS "
    "SELECT * FROM read_parquet(?);",
    [str(GOLD_DIR / "sales_over_time.parquet")]
)

con.execute(
    "CREATE OR REPLACE TABLE sales_by_product AS "
    "SELECT * FROM read_parquet(?);",
    [str(GOLD_DIR / "sales_by_product.parquet")]
)

# Quick sanity check – no pandas/numpy needed
print("Tables:", con.execute("SHOW TABLES;").fetchall())
print("Sample sales_by_customer rows:")
print(con.execute("SELECT * FROM sales_by_customer LIMIT 5;").fetchall())

con.close()

# Note: using fetchall() to avoid extra pandas/numpy dependency; Polars is used for DataFrame work.

DuckDB path: /Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/sales.duckdb
Tables: [('sales_by_customer',), ('sales_by_geo',), ('sales_by_product',), ('sales_over_time',)]
Sample sales_by_customer rows:
[(71, 'Nicholas Craig', 'Zwettl-Niederösterreich', 'Papua New Guinea', 'OCEANIA', 1056, 42), (35, 'Jameson Glover', 'Belgaum', 'N. Mariana Islands', 'OCEANIA', 970, 38), (396, 'Gabriel Cabrera', 'Labico', 'Burundi', 'SUB-SAHARAN AFRICA', 930, 42), (236, 'Allistair Warner', 'Marcq-en-Baroeul', 'Kenya', 'SUB-SAHARAN AFRICA', 929, 43), (291, 'Demetrius Heath', 'Trochu', 'Italy', 'WESTERN EUROPE', 928, 44)]


# 🐧 Master Wide Table – Full Sales Dataset for Dashboards

In [195]:
# Master wide table: join ALL raw fields from bronze layer

import polars as pl

# 1) Load the bronze Parquet tables
countries_bronze = pl.read_parquet(BRONZE_DIR / "countries.parquet")
customers_bronze = pl.read_parquet(BRONZE_DIR / "customers.parquet")
orders_bronze    = pl.read_parquet(BRONZE_DIR / "orders.parquet")
products_bronze  = pl.read_parquet(BRONZE_DIR / "products.parquet")
sales_bronze     = pl.read_parquet(BRONZE_DIR / "sales.parquet")

# 2) Light cleaning / renaming

orders_clean = orders_bronze.rename({"Date": "OrderDate"})

customers_clean = customers_bronze.rename({
    "Name": "CustomerName",
    "Country": "CustomerCountryCode",   # keep the country code here
})

products_clean = products_bronze.rename({
    "Name": "ProductName",
})

countries_clean = countries_bronze.rename({
    "Country": "CountryCode",
    "Name": "CountryName",
    "Area (sq. mi.)": "AreaSqMi",
    "Pop. Density (per sq. mi.)": "PopDensityPerSqMi",
    "Coastline (coast per area ratio)": "CoastlineRatio",
    "Net migration": "NetMigration",
    "Infant mortality (per 1000 births)": "InfantMortalityPer1000",
    "GDP ($ per capita)": "GdpPerCapita",
    "Literacy (%)": "LiteracyPct",
    "Phones (per 1000)": "PhonesPer1000",
    "Arable (%)": "ArablePct",
    "Crops (%)": "CropsPct",
    "Other (%)": "OtherPct",
    "Agriculture": "AgricultureShare",
    "Industry": "IndustryShare",
    "Service": "ServiceShare",
})

# 3) Build the master wide table

master_sales = (
    sales_bronze
    # sales + orders
    .join(orders_clean, on="OrderId", how="left")
    # + customers
    .join(customers_clean, on="CustomerId", how="left")
    # + products
    .join(products_clean, on="ProductId", how="left")
    # + countries: we join on CustomerCountryCode -> CountryCode
    # we do NOT need CountryCode in the final table, because we already have CustomerCountryCode
    .join(
        countries_clean.select([
            "CountryCode",          # join key (internal)
            "CountryName",
            "Currency",
            "Region",
            "Population",
            "AreaSqMi",
            "PopDensityPerSqMi",
            "CoastlineRatio",
            "NetMigration",
            "InfantMortalityPer1000",
            "GdpPerCapita",
            "LiteracyPct",
            "PhonesPer1000",
            "ArablePct",
            "CropsPct",
            "OtherPct",
            "Climate",
            "Birthrate",
            "Deathrate",
            "AgricultureShare",
            "IndustryShare",
            "ServiceShare",
        ]),
        left_on="CustomerCountryCode",
        right_on="CountryCode",
        how="left",
    )
    # 4) Put columns in a logical order (note: we use CustomerCountryCode, not CountryCode)
    .select([
        # core grain
        "SaleId",
        "OrderId",
        "OrderDate",
        "CustomerId",
        "ProductId",
        "Quantity",

        # customer
        "CustomerName",
        "Address",
        "City",
        "CustomerCountryCode",   # this is your country code
        "Email",
        "Active",

        # product
        "ProductName",
        "ManufacturedCountry",
        "WeightGrams",

        # country / macro
        "CountryName",
        "Currency",
        "Region",
        "Population",
        "AreaSqMi",
        "PopDensityPerSqMi",
        "CoastlineRatio",
        "NetMigration",
        "InfantMortalityPer1000",
        "GdpPerCapita",
        "LiteracyPct",
        "PhonesPer1000",
        "ArablePct",
        "CropsPct",
        "OtherPct",
        "Climate",
        "Birthrate",
        "Deathrate",
        "AgricultureShare",
        "IndustryShare",
        "ServiceShare",
    ])
)

print("master_sales shape:", master_sales.shape)
master_sales.head()

master_sales shape: (61883, 36)


SaleId,OrderId,OrderDate,CustomerId,ProductId,Quantity,CustomerName,Address,City,CustomerCountryCode,Email,Active,ProductName,ManufacturedCountry,WeightGrams,CountryName,Currency,Region,Population,AreaSqMi,PopDensityPerSqMi,CoastlineRatio,NetMigration,InfantMortalityPer1000,GdpPerCapita,LiteracyPct,PhonesPer1000,ArablePct,CropsPct,OtherPct,Climate,Birthrate,Deathrate,AgricultureShare,IndustryShare,ServiceShare
i64,i64,str,i64,i64,i64,str,str,str,str,str,bool,str,str,i64,str,str,str,i64,i64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,1,"""2018-01-01""",181,3,1,"""Peter Diaz""","""P.O. Box 106, 4777 Tincidunt A…","""Camrose""","""MX""","""diam.luctus@interdum.com""",true,"""Vaguee""","""CA""",103,"""Mexico""","""MXN""","""LATIN AMER. & CARIB""",107449525,1972550,54.5,0.47,-4.87,20.91,9000,92.2,181.6,12.99,1.31,85.7,1.5,20.69,4.74,0.038,0.259,0.702
2,1,"""2018-01-01""",181,7,9,"""Peter Diaz""","""P.O. Box 106, 4777 Tincidunt A…","""Camrose""","""MX""","""diam.luctus@interdum.com""",true,"""Nebuloon""","""CH""",118,"""Mexico""","""MXN""","""LATIN AMER. & CARIB""",107449525,1972550,54.5,0.47,-4.87,20.91,9000,92.2,181.6,12.99,1.31,85.7,1.5,20.69,4.74,0.038,0.259,0.702
3,2,"""2018-01-01""",119,5,12,"""Stewart Cain""","""Ap #197-8040 Arcu. Rd.""","""Maria""","""AZ""","""eros.Nam.consequat@ornareliber…",true,"""Fuzzi""","""SG""",83,"""Azerbaijan""","""AZN""","""C.W. OF IND. STATES""",7961619,86600,91.9,0.0,-4.9,81.74,3400,97.0,137.1,19.63,2.71,77.66,1.0,20.74,9.75,0.141,0.457,0.402
4,2,"""2018-01-01""",119,10,3,"""Stewart Cain""","""Ap #197-8040 Arcu. Rd.""","""Maria""","""AZ""","""eros.Nam.consequat@ornareliber…",true,"""Uncleary""","""PE""",93,"""Azerbaijan""","""AZN""","""C.W. OF IND. STATES""",7961619,86600,91.9,0.0,-4.9,81.74,3400,97.0,137.1,19.63,2.71,77.66,1.0,20.74,9.75,0.141,0.457,0.402
5,2,"""2018-01-01""",119,4,3,"""Stewart Cain""","""Ap #197-8040 Arcu. Rd.""","""Maria""","""AZ""","""eros.Nam.consequat@ornareliber…",true,"""Dubioum""","""PR""",77,"""Azerbaijan""","""AZN""","""C.W. OF IND. STATES""",7961619,86600,91.9,0.0,-4.9,81.74,3400,97.0,137.1,19.63,2.71,77.66,1.0,20.74,9.75,0.141,0.457,0.402


# 🧱 Save the Master

In [196]:
# Save master wide table for BI tools (Looker Studio, etc.)

MASTER_PATH_PARQUET = GOLD_DIR / "sales_master_flat.parquet"
MASTER_PATH_CSV     = GOLD_DIR / "sales_master_flat.csv"

master_sales.write_parquet(MASTER_PATH_PARQUET)
master_sales.write_csv(MASTER_PATH_CSV)

MASTER_PATH_PARQUET, MASTER_PATH_CSV


(PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/gold/sales_master_flat.parquet'),
 PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/gold/sales_master_flat.csv'))

# 🏗️ Architecture Mapping to Production Stack

This notebook simulates the kind of medallion architecture we would run in Databricks.

---

### Bronze – Landing layer

- Raw JSON files are loaded from `data/*.json`.
- We persist them as columnar Parquet files under `data/bronze/`.
- In production, this would correspond to raw PySpark tables in Databricks backed by object storage (e.g., ADLS) and governed via Unity Catalog.

---

### Silver – Cleaned dimensional model

- We build business-friendly **dimension tables**:
  - `dim_country`
  - `dim_customer`
  - `dim_product`
- We build a **fact table**:
  - `fact_sales_enriched` (orders + customers + products + geography).
- We also create a **wide analytics mart**:
  - `sales_master_wide` – one row per sale with every field the business cares about  
    (customer, product, order date, country, and all country KPI attributes).
- These are written as Parquet under `data/silver/`.
- In production, these would be curated PySpark tables (Unity Catalog) used by downstream analytics and reporting.

---

### Gold – Aggregates & BI outputs

- We create aggregate tables for the dashboard:
  - `sales_by_customer`
  - `sales_by_geo`
  - `sales_over_time`
  - `sales_by_product`
- We also export the **wide** table for BI tools:
  - `sales_master_wide.csv` under `data/gold/` for use in Power BI / Looker Studio.
- These are stored as both Parquet and CSV under `data/gold/` and are what BI tools connect to.
- In production, these would be lightweight serving tables or views optimized for BI tools.

---

### Warehouse & Orchestration

- For this exercise, we register the gold tables in a local DuckDB database (`data/sales.duckdb`) to simulate a small analytics warehouse.
- In your environment, these jobs would be orchestrated and monitored with **Prefect**, scheduled (e.g., daily) to refres
